# Self-Organized Criticality in Grid Congestion

Power grids are a textbook candidate for self-organized criticality (SOC): a system driven by many small, independent decisions (bids, dispatch, flows) that spends most of its time near a critical point, punctuated by occasional large cascading events &ndash; the same statistical signature as sandpiles, forest fires, and earthquakes.

This article looks for that signature in Europe's day-ahead congestion data: do congestion events (a border pinned at its interconnector limit) follow a heavy-tailed distribution in size, the way an avalanche process would, or are they closer to an exponential/Poisson process with no special structure at the large end? We look at event size three ways &ndash; how many 15-minute market time units (MTUs) an event spans, how many hours, and how much congestion rent (EUR) it generates &ndash; plus a first look at day-ahead price levels and price jumps as a related, finer-grained signal.

The easy half of that question gets a clean answer early on. The harder half &ndash; heavy-tailed because it's a genuine scale-free/critical process, or heavy-tailed for some other reason &ndash; turns out to need several more cuts at the data, across different event types, market mechanisms, and finally two tests that pose the question a different way entirely. That's where this notebook spends most of its time.


In [ ]:
# common.py is shared across every article and changes often; without this a
# kernel that already imported it keeps serving the stale module.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent))

from common import (
    border_ccr_mechanism,
    border_clustering_test,
    ccdf_fit_chart,
    congestion_events,
    distribution_histogram,
    fetch_flows_range,
    fit_distribution,
    fit_results_table,
    price_spike_events,
    save_chart,
    scaling_relation,
    scaling_relation_chart,
    zone_prices,
)

## Pulling a year of flow data

A full year, at native (15-minute) resolution, across every border &ndash; enough observations for the tail of an event-size distribution to be meaningful, and enough days for the rarer large events to actually show up. The window ends 10 June 2026, the day Core's Advanced Hybrid Coupling (AHC) went live, keeping the whole sample inside one stable market-structure regime rather than straddling that change.


In [ ]:
date_from = "2025-06-09"
date_to = "2026-06-10"

flows = fetch_flows_range(date_from, date_to)
print(f"{len(flows):,} observations, {flows['border'].nunique()} borders, {date_from} to {date_to}")
flows.head()

## Building congestion events

VoltPath's `GET /api/v1/flows` classifies every MTU into `Convergence`, `Congestion`, `Counter-intuitive`, or `Outage`. A single flagged MTU isn't an "event" in the SOC sense &ndash; we collapse consecutive `Congestion` MTUs on the same border into one contiguous event, the same way a sandpile avalanche is one cascade, not one grain.

UA_IPS borders are dropped &ndash; Ukraine's wartime grid conditions and emergency EU synchronisation drive those numbers, not ordinary market congestion.


In [ ]:
events = congestion_events(flows)
print(f"{len(events):,} congestion events across {events['border'].nunique()} borders")
events.sort_values("duration_h", ascending=False).head()

## Event-size distributions

Three views of the same events: by MTU count (the most literal "avalanche size" &ndash; number of discrete toppled bars), by duration, by congestion rent (EUR, price-gap weighted &ndash; a loosely-defined congestion flag with a near-zero price gap is a very different animal from one with a massive price gap, which neither MTU count nor duration can distinguish), and by physical volume (MWh). Log y-axis throughout &ndash; a heavy tail is invisible on a linear count axis.


In [ ]:
fig = distribution_histogram(events["n_bars"], "Congestion events by size (MTU count)", "MTUs per event")
save_chart(fig, "congestion-events-by-mtu-count")
fig.show()

In [ ]:
fig = distribution_histogram(events["duration_h"], "Congestion events by duration", "Duration (hours)")
save_chart(fig, "congestion-events-by-duration")
fig.show()

In [ ]:
fig = distribution_histogram(events["value_eur"], "Congestion events by rent", "Congestion rent (EUR)")
save_chart(fig, "congestion-events-by-rent")
fig.show()

In [ ]:
fig = distribution_histogram(events["volume_mwh"], "Congestion events by volume", "Volume (MWh)")
save_chart(fig, "congestion-events-by-volume")
fig.show()

## Day-ahead price distribution

A single pooled distribution across every zone and the full year &ndash; day-ahead price is zonal and positive-or-negative, not an event size, so this is a baseline sanity check rather than a power-law candidate in its own right.


In [ ]:
prices = zone_prices(flows)
fig = distribution_histogram(
    prices["price"], "Day-ahead price, all zones pooled", "Price (EUR/MWh)", log_y=False, nbins=250
)
save_chart(fig, "day-ahead-price-distribution")
fig.show()

### Price jumps

The point-to-point `|Δprice|` between consecutive MTUs, per zone, pooled &ndash; a finer-grained, duration-free cut at the same "sudden large change" question event size asks at the congestion level.


In [ ]:
price_jumps = prices.sort_values(["zone", "datetime"]).groupby("zone")["price"].diff().abs().dropna()
fig = distribution_histogram(price_jumps, "Day-ahead price jumps, all zones pooled", "|Δprice| (EUR/MWh)")
save_chart(fig, "day-ahead-price-jump-distribution")
fig.show()

## Fitting candidate distributions

A histogram alone can't distinguish a genuine heavy-tailed (power-law) process from a lognormal or exponential one with a long-looking tail - that distinction needs a proper fit, not an eyeballed log-log plot. We use the Clauset-Shalizi-Newman method: maximum-likelihood estimation of the power-law exponent (`alpha`) and lower cutoff (`xmin`), then a likelihood-ratio test against lognormal and exponential alternatives (`R` > 0 favors power law, `p` < 0.05 means the comparison is decisive; `p` > 0.05 means the two candidates fit indistinguishably well on this data).

Day-ahead price itself isn't fit here - it's a level, not an event magnitude, and goes negative (oversupply/curtailment pricing), which breaks the positivity the method requires. The price jump distribution above stands in as the price-side avalanche analogue instead.

Each distribution has well over 100,000 points; the xmin search inside the MLE fit scales worse than linearly in n, so each one is fit on a fixed-seed random subsample of 50,000 points rather than the full set. `alpha` and `xmin` stabilize well before that sample size, so this trades nothing material for a fit that finishes in seconds instead of minutes per distribution.

In [ ]:
SAMPLE_SIZE = 50_000  # alpha/xmin stabilize well before this; keeps each fit to ~30-40s
price_jumps_nonzero = price_jumps[price_jumps > 0]

fit_results = [
    fit_distribution(events["n_bars"], "Congestion events (MTU count)", discrete=True, subsample=SAMPLE_SIZE),
    fit_distribution(events["duration_h"], "Congestion events (duration, h)", subsample=SAMPLE_SIZE),
    fit_distribution(events["value_eur"], "Congestion events (rent, EUR)", subsample=SAMPLE_SIZE),
    fit_distribution(events["volume_mwh"], "Congestion events (volume, MWh)", subsample=SAMPLE_SIZE),
    fit_distribution(price_jumps_nonzero, "Day-ahead price jumps (EUR/MWh)", subsample=SAMPLE_SIZE),
]
fit_results_table(fit_results)

## CCDF fits

Empirical complementary CDF (log-log) against all three candidate fits, for each distribution above.

In [ ]:
for result in fit_results:
    fig = ccdf_fit_chart(result["fit"], result["name"], result["name"])
    slug = result["name"].lower().split("(")[0].strip().replace(" ", "-")
    save_chart(fig, f"ccdf-fit-{slug}")
    fig.show()

## Summary

Every distribution decisively rules out an exponential (thin-tailed, memoryless) process - the heavy tail is real, not an artifact. Power law vs lognormal is the harder call: only day-ahead price jumps give a clean, decisive answer; the four congestion-event metrics are statistically indistinguishable between the two.

The rest of this notebook keeps pushing on that harder call: different event types, different market mechanisms, a structural break at a known transition, and - once none of those settle it either - two tests that ask the avalanche question in a completely different way.


In [ ]:
summary = fit_results_table(fit_results)[
    ["xmin", "alpha", "n_tail", "ks_power_law", "ks_lognormal", "ks_exponential", "verdict"]
].round(4)
summary

## Counter-intuitive flow events

Counter-intuitive flows &ndash; power moving against the price gradient &ndash; are a different physical mechanism from NTC-pinning congestion (loop flow through the meshed grid rather than a border sitting at its allocated limit), so they get their own event construction and fit rather than being folded into the congestion numbers above.


In [ ]:
events_ci = congestion_events(flows, flow_type="Counter-intuitive")
print(f"{len(events_ci):,} counter-intuitive events, {events_ci['border'].nunique()} borders")

ci_fit_results = [
    fit_distribution(events_ci["n_bars"], "Counter-intuitive events (MTU count)", discrete=True),
    fit_distribution(events_ci["duration_h"], "Counter-intuitive events (duration, h)"),
    fit_distribution(events_ci["value_eur"], "Counter-intuitive events (rent, EUR)"),
    fit_distribution(events_ci["volume_mwh"], "Counter-intuitive events (volume, MWh)"),
]
fit_results_table(ci_fit_results)

In [ ]:
for result in ci_fit_results:
    fig = ccdf_fit_chart(result["fit"], result["name"], result["name"])
    slug = result["name"].lower().replace(" ", "-").translate(str.maketrans("", "", ",()")).replace("/", "-")
    save_chart(fig, f"ccdf-fit-{slug}")
    fig.show()

## Price spike episodes

A complementary view to the raw price distribution and point-to-point jumps above: contiguous runs where a zone's price sits more than 3 standard deviations above that zone's own full-sample mean &ndash; the same "mean + k&sigma;" rule used in the day-ahead price-spike literature, with a trailing window long enough (one year) that it coincides with the full sample. This gives a count/duration/magnitude triple directly comparable in spirit to the congestion events above, but driven by price level rather than a flow classifier.


In [ ]:
prices_for_spikes = zone_prices(flows)
price_spikes = price_spike_events(prices_for_spikes)
print(f"{len(price_spikes):,} price-spike episodes, {price_spikes['zone'].nunique()} zones")

spike_fit_results = [
    fit_distribution(price_spikes["n_bars"], "Price spikes (MTU count)", discrete=True),
    fit_distribution(price_spikes["duration_h"], "Price spikes (duration, h)"),
    fit_distribution(price_spikes["excess_eur"], "Price spikes (excess, EUR/MWh)"),
]
fit_results_table(spike_fit_results)

In [ ]:
for result in spike_fit_results:
    fig = ccdf_fit_chart(result["fit"], result["name"], result["name"])
    slug = result["name"].lower().replace(" ", "-").translate(str.maketrans("", "", ",()")).replace("/", "-")
    save_chart(fig, f"ccdf-fit-{slug}")
    fig.show()

## Topology: meshed vs radial CCRs

If the heavy tail is a genuine scale-free/cascade signature rather than an artifact of how any one border happens to be operated, it should show up differently depending on grid topology: Core's flow-based coupling clears the whole meshed region jointly (cascades can propagate border to border), while the price-coupled NTC regions (SWE/GRIT/SEE/Baltic, and separately Hansa/Nordic) allocate each border's capacity independently of its neighbors (a radial/lineset structure with fewer paths for a local event to propagate along). Congestion rent (`value_eur`) is refit per topology bucket to test whether the power-law exponent or cutoff scale differs.


In [ ]:
TOPOLOGY_BUCKETS = {
    "Core (flow-based)": ["Core (price-coupled, flow-based)"],
    "NTC, radial": [
        "SWE (price-coupled NTC, radial)",
        "GRIT (price-coupled NTC, radial)",
        "SEE (price-coupled NTC, radial)",
        "Baltic (price-coupled NTC, radial)",
    ],
    "NTC, lineset": [
        "Hansa (price-coupled NTC, lineset)",
        "Nordic (price-coupled NTC, lineset)",
    ],
}

events_mechanism = events["border"].map(border_ccr_mechanism)
topology_fit_results = []
for bucket_name, mechanisms in TOPOLOGY_BUCKETS.items():
    bucket_events = events[events_mechanism.isin(mechanisms)]
    topology_fit_results.append(
        fit_distribution(bucket_events["value_eur"], f"Congestion rent - {bucket_name}")
    )
topology_table = fit_results_table(topology_fit_results)
topology_table

In [ ]:
LOW_TAIL_WARNING_N = 100
for result in topology_fit_results:
    if result["n_tail"] < LOW_TAIL_WARNING_N:
        print(
            f"{result['name']}: n_tail={result['n_tail']} < {LOW_TAIL_WARNING_N} - "
            "xmin/alpha point estimate only, LR-test comparisons not trustworthy at this n"
        )
    fig = ccdf_fit_chart(result["fit"], result["name"], "Congestion rent (EUR)")
    slug = result["name"].lower().replace(" ", "-").translate(str.maketrans("", "", ",()")).replace("/", "-")
    save_chart(fig, f"ccdf-fit-{slug}")
    fig.show()

## Structural shift: Core Advanced Hybrid Coupling go-live

Core's Advanced Hybrid Coupling (AHC) went live on 10 June 2026, moving the region's external borders (to the Nordics, Baltics, and a few others) from a static day-ahead forecast to a jointly solved virtual bidding zone. This is a separate, narrower pull bracketing the go-live date &ndash; the main window above deliberately ends at go-live so it never crosses this transition, but testing for a shift *at* the transition needs data on both sides of it. Scoped to Core-mechanism borders only, and the go-live week itself (6&ndash;14 June) is dropped from both sides to avoid transition-week noise contaminating either half.


In [ ]:
CORE_AHC_EVENT = {
    "name": "Core Advanced Hybrid Coupling",
    "date": "2026-06-10",
    "date_from": "2026-06-06",
    "date_to": "2026-06-14",
}

ahc_window_from = (pd.Timestamp(CORE_AHC_EVENT["date"]) - pd.Timedelta(days=90)).strftime("%Y-%m-%d")
ahc_window_to = (pd.Timestamp(CORE_AHC_EVENT["date"]) + pd.Timedelta(days=90)).strftime("%Y-%m-%d")
flows_ahc = fetch_flows_range(ahc_window_from, ahc_window_to)
flows_ahc_core = flows_ahc[
    flows_ahc["border"].map(border_ccr_mechanism) == "Core (price-coupled, flow-based)"
]
print(
    f"{len(flows_ahc_core):,} Core-border observations, "
    f"{flows_ahc_core['border'].nunique()} borders, {ahc_window_from} to {ahc_window_to}"
)

In [ ]:
events_ahc = congestion_events(flows_ahc_core)
events_ahc_before = events_ahc[events_ahc["end"] < CORE_AHC_EVENT["date_from"]]
events_ahc_after = events_ahc[events_ahc["start"] >= CORE_AHC_EVENT["date_to"]]
print(f"{len(events_ahc_before):,} events before go-live, {len(events_ahc_after):,} after")

ahc_fit_results = [
    fit_distribution(events_ahc_before["value_eur"], "Core rent - before AHC"),
    fit_distribution(events_ahc_after["value_eur"], "Core rent - after AHC"),
    fit_distribution(events_ahc_before["n_bars"], "Core MTU count - before AHC", discrete=True),
    fit_distribution(events_ahc_after["n_bars"], "Core MTU count - after AHC", discrete=True),
]
fit_results_table(ahc_fit_results)

In [ ]:
for result in ahc_fit_results:
    fig = ccdf_fit_chart(result["fit"], result["name"], result["name"])
    slug = result["name"].lower().replace(" ", "-").replace(",", "").replace("/", "-")
    save_chart(fig, f"ccdf-fit-{slug}")
    fig.show()

## Why doesn't power law ever win outright?

Across every distribution fit so far - five in Phase 1, seven more in Phase 2, sixteen fits total - the likelihood-ratio tests never once favor a plain power law over lognormal. The exponential is dead everywhere (the heavy tail is real), but "heavy-tailed" keeps landing on lognormal or inconclusive, never on the sharp scale-free signature SOC predicts.

Two explanations, not mutually exclusive:

1. **Statistical**: power-law-vs-lognormal is a genuinely hard discrimination problem on any finite-range, finite-sample tail - Clauset, Shalizi & Newman's own 2009 paper says as much. Maybe there's nothing more to learn here; the data just isn't decisive enough.
2. **Mechanistic**: every variable tested so far is physically bounded - an event can't outlast a market day, congestion gets actively mitigated, price is regulatory-capped. An *unbounded* power law may be the wrong null hypothesis for a bounded system; the bounded-system SOC prediction is a power-law body with an exponential cutoff (a truncated power law), not what's been tested yet.

The next three cuts use data already built above - no new API pulls - to test these from three different angles: does an exponential cutoff actually fit better (explanation 2, directly); does event size scale with duration the way an avalanche process predicts (a signature between two variables, not read off one marginal distribution); and do large events on one border coincide with large events on a neighboring border more than chance would predict (a direct test of cascade/propagation, the mechanism SOC actually claims, as opposed to just "heavy-tailed").

### Does an exponential cutoff help? (truncated power law)

A 4th CSN candidate alongside power law / lognormal / exponential: `alpha * x**-alpha * exp(-Lambda*x)`, nesting the plain power law as the special case `Lambda=0`. If every bounded variable tested is better described with `Lambda > 0` than without, that's direct evidence for the mechanistic explanation above - the ceiling on event size is doing real statistical work, not just common sense.

In [ ]:
all_fit_results = fit_results + ci_fit_results + spike_fit_results + topology_fit_results + ahc_fit_results
truncation_table = fit_results_table(all_fit_results)[
    [
        "alpha",
        "ks_power_law",
        "ks_truncated_power_law",
        "r_truncated_power_law",
        "p_truncated_power_law",
        "truncation",
    ]
].round(4)
truncation_table

### Does size scale with duration the way an avalanche would?

SOC's size-duration scaling relation (`size ~ duration^gamma`) is a stronger, more specific signature than any single marginal distribution's tail - it asks whether two different measurements of the same event move together the way a cascading/compounding process predicts, rather than independently. Tested on every event set that carries both a size and a duration column; `n_bars` is dropped as a "size" here since it's a near-exact linear multiple of `duration_h` for fixed-cadence borders, not an independent measurement.

In [ ]:
scaling_results = [
    scaling_relation(events["value_eur"], events["duration_h"], "Congestion: rent vs duration"),
    scaling_relation(events["volume_mwh"], events["duration_h"], "Congestion: volume vs duration"),
    scaling_relation(events_ci["value_eur"], events_ci["duration_h"], "Counter-intuitive: rent vs duration"),
    scaling_relation(
        events_ci["volume_mwh"], events_ci["duration_h"], "Counter-intuitive: volume vs duration"
    ),
    scaling_relation(
        price_spikes["excess_eur"], price_spikes["duration_h"], "Price spikes: excess vs duration"
    ),
]
scaling_table = pd.DataFrame(scaling_results).set_index("name").round(4)
scaling_table

In [ ]:
for result in scaling_results:
    size_col, duration_col, df = {
        "Congestion: rent vs duration": ("value_eur", "duration_h", events),
        "Congestion: volume vs duration": ("volume_mwh", "duration_h", events),
        "Counter-intuitive: rent vs duration": ("value_eur", "duration_h", events_ci),
        "Counter-intuitive: volume vs duration": ("volume_mwh", "duration_h", events_ci),
        "Price spikes: excess vs duration": ("excess_eur", "duration_h", price_spikes),
    }[result["name"]]
    fig = scaling_relation_chart(df[size_col], df[duration_col], result, result["name"], size_col)
    slug = result["name"].lower().replace(" ", "-").translate(str.maketrans("", "", ",():"))
    save_chart(fig, f"scaling-{slug}")
    fig.show()

### Do large events cluster across neighboring borders?

The most direct test of what SOC actually claims - cascade/propagation, not just "the tail is heavy." Adjacent borders are those sharing a zone endpoint (e.g. `DE -> FR` and `FR -> CH` share `FR`). The null is a circular shift of each border's own event timeline within the study window: it keeps each border's event count and internal spacing intact but scrambles the alignment between borders, so a common external driver hitting many borders at once (a continent-wide demand or generation shock) does not by itself inflate the observed co-occurrence count - only genuine propagation between neighbors does. Restricted to the top quartile of congestion events by rent, since cascade propagation is specifically a tail phenomenon.

In [ ]:
clustering_result = border_clustering_test(events, lag_h=6.0, n_perm=1000, quantile=0.75, seed=0)
clustering_result